<a href="https://colab.research.google.com/github/missturlubayeva/IST3134-Assignment-2/blob/main/IST3134.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/tracks_features_3m.csv')

In [ ]:
df.info()

In [ ]:
df[['danceability', 'energy', 'acousticness', 'valence', 'tempo']].describe()

,danceability,energy,acousticness,valence,tempo
count,3.612075e+06,3.612075e+06,3.612075e+06,3.612075e+06,3.612075e+06
mean,4.930565e-01,5.095363e-01,4.467511e-01,4.279866e-01,1.176344e+02
std,1.896694e-01,2.946838e-01,3.852013e-01,2.704845e-01,3.093704e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.560000e-01,2.520000e-01,3.760000e-02,1.910000e-01,9.405400e+01
50%,5.010000e-01,5.240000e-01,3.890000e-01,4.030000e-01,1.167260e+02
75%,6.330000e-01,7.660000e-01,8.610000e-01,6.440000e-01,1.370460e+02
max,1.000000e+00,1.000000e+00,9.960000e-01,1.000000e+00,2.489340e+02


In [ ]:
import pandas as pd
import time

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# -----------------------------
# Start timer
# -----------------------------
start_time = time.time()

In [ ]:
# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/tracks_features_3m.csv')

load_time = time.time()

In [ ]:
# -----------------------------
# Select features
# -----------------------------
features = [
    "danceability",
    "energy",
    "acousticness",
    "valence",
    "tempo"
]

X = df[features]

# Remove missing values if any
X = X.dropna()

In [ ]:
# -----------------------------
# Standardisation
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

preprocess_time = time.time()

In [ ]:
# -----------------------------
# K-Means Clustering
# -----------------------------
k = 2

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

training_time = time.time()


In [ ]:
# -----------------------------
# Add cluster labels to the original features
# -----------------------------
result_df = X.copy()
result_df["cluster"] = clusters

In [ ]:
sample_df = result_df.sample(n=100000, random_state=42)

sample_X = scaler.transform(
    sample_df[features]
)

sample_clusters = sample_df["cluster"]

silhouette = silhouette_score(
    sample_X,
    sample_clusters
)

In [ ]:
# -----------------------------
# Add cluster labels
# -----------------------------
result_df = X.copy()
result_df["cluster"] = clusters

# -----------------------------
# Cluster statistics
# -----------------------------
cluster_summary = (
    result_df
    .groupby("cluster")
    .agg({
        "danceability": ["mean", "min", "max"],
        "energy": ["mean", "min", "max"],
        "acousticness": ["mean", "min", "max"],
        "valence": ["mean", "min", "max"],
        "tempo": ["mean", "min", "max"]
    })
)

# -----------------------------
# Cluster counts
# -----------------------------
cluster_counts = (
    result_df["cluster"]
    .value_counts()
    .sort_index()
)


In [ ]:
# -----------------------------
# Timing results
# -----------------------------
end_time = time.time()

print("=" * 50)
print("PANDAS + SCIKIT-LEARN RESULTS")
print("=" * 50)

print(f"Loading Time:       {load_time - start_time:.2f} sec")
print(f"Preprocessing Time: {preprocess_time - load_time:.2f} sec")
print(f"K-Means Time:       {training_time - preprocess_time:.2f} sec")
print(f"Total Time:         {end_time - start_time:.2f} sec")

print("\nSilhouette Score:")
print(round(silhouette, 4))

print("\nCluster Counts:")
print(cluster_counts)

print("\nCluster Summary:")
print(cluster_summary)

PANDAS + SCIKIT-LEARN RESULTS
Loading Time:       51.73 sec
Preprocessing Time: 101.37 sec
K-Means Time:       31.76 sec
Total Time:         1187.00 sec

Silhouette Score:
0.3306

Cluster Counts:
cluster
0    2183646
1    1428429
Name: count, dtype: int64

Cluster Summary:
        danceability                   energy               acousticness       \
                mean     min    max      mean      min  max         mean  min   
cluster                                                                         
0           0.562443  0.0343  1.000  0.700127  0.00002  1.0     0.198827  0.0   
1           0.386985  0.0000  0.987  0.218179  0.00000  1.0     0.825753  0.0   

                 valence                 tempo                   
           max      mean  min  max        mean     min      max  
cluster                                                          
0        0.996  0.538754  0.0  1.0  124.900982  30.133  248.934  
1        0.996  0.258656  0.0  1.0  106.525820   0.000  